In [0]:
import pyspark.sql.functions as F

# 1. Leer la tabla Delta creada en Bronze
bronze_df = spark.table("bronze_inventory_raw")

# 2. Aplicar reglas de calidad y limpieza para inventario activo
silver_df = (
    bronze_df
    # Filtrar SKUs no nulos y precios mayores a cero
    .filter(F.col("StockCode").isNotNull() & (F.col("UnitPrice") > 0))
    # Filtrar devoluciones o cantidades negativas (inventario activo/transaccionado)
    .filter(F.col("Quantity") > 0)
    # Crear columna de valor total transaccionado por registro
    .withColumn("TotalValue", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
    # Agregar marca de tiempo de ingestión
    .withColumn("ProcessedAt", F.current_timestamp())
)

# 3. Guardar como Tabla Delta Silver
(
    silver_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable("silver_inventory_cleaned")
)

# 4. Verificar resultado
display(spark.table("silver_inventory_cleaned").limit(10))